# 09 — Borough Comparison

Cross-borough analysis that runs **only when 2+ boroughs have been analyzed**.

**Outputs** (to `outputs/Comparison/`):
1. Comparison bar charts — cell counts, class distribution %, accuracy per borough
2. Combined map — all boroughs’ heatmaps on one map
3. Side-by-side per-borough diagrams — heatmaps and dashboards in a single sheet

In [ ]:
# ── Papermill parameters ────────────────────────────────────────
PLOTS_DIR = "outputs/Comparison"
COMBINED_SHEETS = True

In [ ]:
import matplotlib
matplotlib.use("Agg")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.image as mpimg
from matplotlib.colors import LinearSegmentedColormap
from pathlib import Path
import json
import math
import os

# Clean previous comparison outputs so nothing is stale
_plots_path = Path(PLOTS_DIR)
if _plots_path.exists():
    for f in _plots_path.iterdir():
        if f.is_file():
            f.unlink()
_plots_path.mkdir(parents=True, exist_ok=True)

# Clean and create Combined Plots subfolder
_combined_path = _plots_path / "Combined Plots"
if _combined_path.exists():
    for f in _combined_path.iterdir():
        if f.is_file():
            f.unlink()
_combined_path.mkdir(parents=True, exist_ok=True)

with open("grid.json", encoding="utf-8") as f:
    config = json.load(f)

CELL_SIZE_M = config["grid_cell_size_m"]
INCLUDE_OTHER = config.get("include_other", False)
CLASS_ORDER = ["Commercial", "Residential", "Other"] if INCLUDE_OTHER else ["Commercial", "Residential"]
CLASS_COLS = {"Commercial": "#B2182B", "Residential": "#2166AC", "Other": "#1B7837"}

In [ ]:
# ── Discover all borough folders with prediction results ─

csv_root = Path("csv")
_KNOWN_BOROUGHS = {"Manhattan", "Brooklyn", "Queens", "Bronx", "Staten Island"}

boroughs = {}

for folder in sorted(csv_root.iterdir()):
    if folder.is_dir() and (folder / "07_predictions.csv").exists():
        df = pd.read_csv(folder / "07_predictions.csv", dtype={"cell_id": str})
        if len(df) > 0:
            # Folders matching a known borough keep their name;
            # anything else (old combined/timestamped folders) → "Combined"
            label = folder.name if folder.name in _KNOWN_BOROUGHS else "Combined"
            boroughs[label] = {"df": df, "csv_folder": folder}
            print(f"  Found: {label} ({len(df)} cells)")

# Sort: known boroughs alphabetically first, "Combined" always last
boroughs = dict(
    sorted(boroughs.items(),
           key=lambda kv: (0, kv[0]) if kv[0] in _KNOWN_BOROUGHS else (1, kv[0]))
)

print(f"\nTotal entries found: {len(boroughs)}")

if len(boroughs) < 2:
    print("\nSkipping comparison — need at least 2 analyzed boroughs.")
    raise SystemExit(0)

In [ ]:
# ── Build comparison dataframe ───────────────────────

_CLASS_MAP = {
    "Commercial": "Commercial",
    "Mixed-Use": "Residential",
    "Residential": "Residential",
}
if INCLUDE_OTHER:
    _CLASS_MAP.update({
        "Institutional": "Other", "Open Space": "Other",
        "Industrial": "Other", "Infrastructure": "Other",
    })

summary_rows = []
for name, info in boroughs.items():
    df = info["df"]
    # outputs_folder: use actual folder name on disk (may differ from label)
    info["outputs_folder"] = Path(f"outputs/{info['csv_folder'].name}")
    df["actual_class"] = df["zone_type"].map(_CLASS_MAP)
    accuracy = (df["predicted_zone"] == df["actual_class"]).mean()
    row = {"borough": name, "total_cells": len(df), "accuracy": accuracy}
    for cls in CLASS_ORDER:
        n_pred = (df["predicted_zone"] == cls).sum()
        n_actual = (df["actual_class"] == cls).sum()
        row[f"pred_{cls}"] = n_pred
        row[f"pct_pred_{cls}"] = 100 * n_pred / len(df)
        row[f"actual_{cls}"] = n_actual
        row[f"pct_actual_{cls}"] = 100 * n_actual / len(df)
    summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows)
print(df_summary.to_string(index=False))

In [ ]:
# ── Plot 1: Comparison bar charts ────────────────────

n_boroughs = len(df_summary)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(f"Borough Comparison \u2014 {n_boroughs} boroughs \u00b7 {CELL_SIZE_M}m grid",
             fontsize=14, fontweight="bold")

b_names = df_summary["borough"].tolist()
x = np.arange(n_boroughs)

# (0,0) Total cells per borough
ax = axes[0, 0]
ax.bar(x, df_summary["total_cells"], color="#555555", alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(b_names, rotation=30, ha="right", fontsize=9)
ax.set_ylabel("Grid cells")
ax.set_title("Total Grid Cells")
for i, v in enumerate(df_summary["total_cells"]):
    ax.text(i, v + 20, str(v), ha="center", fontsize=9)

# (0,1) Accuracy per borough
ax = axes[0, 1]
colors_acc = ["#2ca02c" if a >= 0.8 else "#ff7f0e" if a >= 0.7 else "#d62728"
              for a in df_summary["accuracy"]]
ax.bar(x, df_summary["accuracy"] * 100, color=colors_acc, alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(b_names, rotation=30, ha="right", fontsize=9)
ax.set_ylabel("Accuracy (%)")
ax.set_title("Overall Accuracy")
ax.set_ylim(0, 105)
for i, v in enumerate(df_summary["accuracy"]):
    ax.text(i, v * 100 + 1, f"{v:.1%}", ha="center", fontsize=9)

# (1,0) Predicted class distribution (stacked %)
ax = axes[1, 0]
bottom = np.zeros(n_boroughs)
for cls in CLASS_ORDER:
    vals = df_summary[f"pct_pred_{cls}"].values
    ax.bar(x, vals, bottom=bottom, color=CLASS_COLS[cls], label=cls, alpha=0.85)
    for i, (v, b) in enumerate(zip(vals, bottom)):
        if v > 5:
            ax.text(i, b + v / 2, f"{v:.0f}%", ha="center", va="center", fontsize=8, color="white", fontweight="bold")
    bottom += vals
ax.set_xticks(x)
ax.set_xticklabels(b_names, rotation=30, ha="right", fontsize=9)
ax.set_ylabel("Percentage")
ax.set_title("Predicted Class Distribution")
ax.legend(fontsize=9)
ax.set_ylim(0, 105)

# (1,1) Actual class distribution (stacked %)
ax = axes[1, 1]
bottom = np.zeros(n_boroughs)
for cls in CLASS_ORDER:
    vals = df_summary[f"pct_actual_{cls}"].values
    ax.bar(x, vals, bottom=bottom, color=CLASS_COLS[cls], label=cls, alpha=0.85)
    for i, (v, b) in enumerate(zip(vals, bottom)):
        if v > 5:
            ax.text(i, b + v / 2, f"{v:.0f}%", ha="center", va="center", fontsize=8, color="white", fontweight="bold")
    bottom += vals
ax.set_xticks(x)
ax.set_xticklabels(b_names, rotation=30, ha="right", fontsize=9)
ax.set_ylabel("Percentage")
ax.set_title("Actual Class Distribution (PLUTO)")
ax.legend(fontsize=9)
ax.set_ylim(0, 105)

plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/01_borough_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {PLOTS_DIR}/01_borough_comparison.png")

In [ ]:
# ── Plot 2: Combined map — all boroughs on one figure ─

import contextily as ctx

# Merge all prediction DataFrames
all_dfs = []
for name, info in boroughs.items():
    df_copy = info["df"].copy()
    df_copy["borough"] = name
    all_dfs.append(df_copy)
df_all = pd.concat(all_dfs, ignore_index=True)

REF_LAT = df_all["cell_lat"].mean()
LAT_STEP = CELL_SIZE_M / 111_000
LON_STEP = CELL_SIZE_M / (111_000 * math.cos(math.radians(REF_LAT)))
HALF_LAT = LAT_STEP / 2
HALF_LON = LON_STEP / 2

# Determine figure aspect ratio from data bounds
lat_range = df_all["cell_lat"].max() - df_all["cell_lat"].min()
lon_range = df_all["cell_lon"].max() - df_all["cell_lon"].min()
aspect = lat_range / (lon_range * math.cos(math.radians(REF_LAT))) if lon_range > 0 else 1.5
fig_w = max(12, min(20, 14))
fig_h = max(8, min(22, fig_w * aspect))

fig, ax = plt.subplots(figsize=(fig_w, fig_h))

if INCLUDE_OTHER:
    for _, row in df_all.iterrows():
        base_color = CLASS_COLS.get(row["predicted_zone"], "#999999")
        conf = row.get("confidence", 0.7)
        alpha = 0.4 + 0.5 * conf
        rect = mpatches.Rectangle(
            (row["cell_lon"] - HALF_LON, row["cell_lat"] - HALF_LAT),
            LON_STEP, LAT_STEP, linewidth=0.05, edgecolor="gray",
            facecolor=base_color, alpha=alpha)
        ax.add_patch(rect)
else:
    cmap = LinearSegmentedColormap.from_list("res_com", ["#2166AC", "#F7F7F7", "#B2182B"])
    for _, row in df_all.iterrows():
        prob_com = row.get("prob_commercial", 0.5)
        rect = mpatches.Rectangle(
            (row["cell_lon"] - HALF_LON, row["cell_lat"] - HALF_LAT),
            LON_STEP, LAT_STEP, linewidth=0.05, edgecolor="gray",
            facecolor=cmap(prob_com), alpha=0.75)
        ax.add_patch(rect)

pad = 0.005
ax.set_xlim(df_all["cell_lon"].min() - pad, df_all["cell_lon"].max() + pad)
ax.set_ylim(df_all["cell_lat"].min() - pad, df_all["cell_lat"].max() + pad)
ax.set_aspect("equal")

try:
    ctx.add_basemap(ax, crs="EPSG:4326",
                    source=ctx.providers.CartoDB.PositronNoLabels,
                    zoom=12, alpha=0.4)
except Exception as e:
    print(f"Basemap download failed: {e}")

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")

# Title with global percentages
counts_all = df_all["predicted_zone"].value_counts()
pct_parts = " \u00b7 ".join(
    f"{cls} {100 * counts_all.get(cls, 0) / len(df_all):.1f}%"
    for cls in CLASS_ORDER if counts_all.get(cls, 0) > 0)

# Per-borough breakdown
borough_pcts = []
for name, info in boroughs.items():
    df_b = info["df"]
    counts_b = df_b["predicted_zone"].value_counts()
    parts = ", ".join(
        f"{cls[:3]} {100 * counts_b.get(cls, 0) / len(df_b):.0f}%"
        for cls in CLASS_ORDER if counts_b.get(cls, 0) > 0)
    borough_pcts.append(f"{name}: {parts}")
borough_line = "  |  ".join(borough_pcts)

ax.set_title(
    f"Combined Heatmap \u2014 {len(df_all)} cells ({CELL_SIZE_M}m grid)\n"
    f"{pct_parts}\n"
    f"{borough_line}",
    fontsize=11)

if INCLUDE_OTHER:
    legend_patches = [mpatches.Patch(color=CLASS_COLS[c], label=c) for c in CLASS_ORDER]
    ax.legend(handles=legend_patches, loc="lower right", fontsize=10,
              framealpha=0.9, edgecolor="gray")
else:
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(0, 1))
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, shrink=0.3, pad=0.02)
    cbar.set_label("P(Commercial)", fontsize=10)

plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/02_combined_map.png", dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved: {PLOTS_DIR}/02_combined_map.png")

In [ ]:
# ── Combined interactive folium map ───────────────────

try:
    import folium
    from folium import Rectangle

    m = folium.Map(
        location=[df_all["cell_lat"].mean(), df_all["cell_lon"].mean()],
        zoom_start=11,
        tiles="CartoDB positron",
    )

    for _, row in df_all.iterrows():
        bounds = [
            [row["cell_lat"] - HALF_LAT, row["cell_lon"] - HALF_LON],
            [row["cell_lat"] + HALF_LAT, row["cell_lon"] + HALF_LON],
        ]
        prob_com = row.get("prob_commercial", 0.5)

        if INCLUDE_OTHER:
            color = CLASS_COLS.get(row["predicted_zone"], "#999999")
            conf = row.get("confidence", 0.7)
            opacity = 0.5 + 0.4 * conf
            popup_text = (f"<b>{row['cell_id']}</b><br>"
                          f"Borough: {row['borough']}<br>"
                          f"Actual: {row['zone_type']}<br>"
                          f"Predicted: <b>{row['predicted_zone']}</b><br>"
                          f"Confidence: {conf:.0%}")
        else:
            color = "#B2182B" if prob_com >= 0.5 else "#2166AC"
            opacity = 0.4 + 0.5 * abs(prob_com - 0.5) * 2
            popup_text = (f"<b>{row['cell_id']}</b><br>"
                          f"Borough: {row['borough']}<br>"
                          f"Actual: {row['zone_type']}<br>"
                          f"Predicted: <b>{row['predicted_zone']}</b><br>"
                          f"P(Commercial): {prob_com:.0%}")

        Rectangle(
            bounds=bounds,
            color="gray", weight=0.2,
            fill=True, fill_color=color, fill_opacity=opacity,
            popup=folium.Popup(popup_text, max_width=200),
        ).add_to(m)

    html_path = f"{PLOTS_DIR}/02_combined_map_interactive.html"
    m.save(html_path)
    print(f"Saved: {html_path}")

except ImportError:
    print("folium not installed \u2014 skipping interactive map.")

In [ ]:
# ── Plot 3: Side-by-side heatmaps per borough ────────

b_names = list(boroughs.keys())

# Load per-borough heatmap PNGs from their latest outputs folder
heatmap_images = {}
for name, info in boroughs.items():
    out_folder = info.get("outputs_folder")
    if out_folder:
        img_path = out_folder / "08_heatmap_predictions.png"
        if img_path.exists():
            heatmap_images[name] = mpimg.imread(str(img_path))

if heatmap_images:
    n_imgs = len(heatmap_images)
    ncols = min(n_imgs, 3)
    nrows = math.ceil(n_imgs / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(8 * ncols, 12 * nrows))
    if n_imgs == 1:
        axes = np.array([axes])
    axes = np.atleast_2d(axes)

    fig.suptitle("Per-Borough Heatmaps", fontsize=16, fontweight="bold")

    for i, (name, img) in enumerate(heatmap_images.items()):
        r, c = divmod(i, ncols)
        ax = axes[r, c]
        ax.imshow(img)
        ax.set_title(name, fontsize=14)
        ax.axis("off")

    # Hide unused subplots
    for j in range(n_imgs, nrows * ncols):
        r, c = divmod(j, ncols)
        axes[r, c].axis("off")

    plt.tight_layout()
    plt.savefig(f"{PLOTS_DIR}/03_heatmaps_side_by_side.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {PLOTS_DIR}/03_heatmaps_side_by_side.png")
else:
    print("No per-borough heatmap PNGs found \u2014 skipping side-by-side heatmaps.")

In [ ]:
# ── Plot 4: Side-by-side dashboards per borough ───────

dashboard_images = {}
for name, info in boroughs.items():
    out_folder = info.get("outputs_folder")
    if out_folder:
        img_path = out_folder / "09_summary_dashboard.png"
        if img_path.exists():
            dashboard_images[name] = mpimg.imread(str(img_path))

if dashboard_images:
    n_imgs = len(dashboard_images)
    ncols = min(n_imgs, 2)
    nrows = math.ceil(n_imgs / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(14 * ncols, 10 * nrows))
    if n_imgs == 1:
        axes = np.array([axes])
    axes = np.atleast_2d(axes)

    fig.suptitle("Per-Borough Dashboards", fontsize=16, fontweight="bold")

    for i, (name, img) in enumerate(dashboard_images.items()):
        r, c = divmod(i, ncols)
        ax = axes[r, c]
        ax.imshow(img)
        ax.set_title(name, fontsize=14)
        ax.axis("off")

    for j in range(n_imgs, nrows * ncols):
        r, c = divmod(j, ncols)
        axes[r, c].axis("off")

    plt.tight_layout()
    plt.savefig(f"{PLOTS_DIR}/04_dashboards_side_by_side.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {PLOTS_DIR}/04_dashboards_side_by_side.png")
else:
    print("No per-borough dashboard PNGs found \u2014 skipping side-by-side dashboards.")

In [ ]:
# ── Plot 5: Feature means comparison across boroughs ──

FEATURE_COLS = ["amenity_density", "shop_density_km2",
                "brand_ratio", "tourism_density", "landuse_entropy",
                "amenity_ratio_food_drink", "avg_floors", "avg_yearbuilt",
                "building_count", "total_bldg_area"]

# Load feature data from combined_grid.csv (predictions CSV lacks features)
_all_dfs = {}
for name, info in boroughs.items():
    combined_path = info["csv_folder"] / "combined_grid.csv"
    if combined_path.exists():
        _all_dfs[name] = pd.read_csv(combined_path, dtype={"cell_id": str})
    else:
        print(f"  {name}: no combined_grid.csv — skipping")

available_feats = [f for f in FEATURE_COLS if all(f in df.columns for df in _all_dfs.values())]

if available_feats and len(_all_dfs) >= 2:
    feat_means = {}
    for name, df in _all_dfs.items():
        feat_means[name] = df[available_feats].mean()
    df_feats = pd.DataFrame(feat_means).T

    # Normalize each feature to [0,1] for comparison
    df_norm = (df_feats - df_feats.min()) / (df_feats.max() - df_feats.min() + 1e-9)

    fig, ax = plt.subplots(figsize=(14, max(6, len(available_feats) * 0.6)))
    y_pos = np.arange(len(available_feats))
    bar_h = 0.8 / len(_all_dfs)
    cmap_boroughs = plt.cm.Set2(np.linspace(0, 1, len(_all_dfs)))

    for i, (name, row) in enumerate(df_norm.iterrows()):
        ax.barh(y_pos + i * bar_h, row[available_feats].values,
                bar_h, label=name, color=cmap_boroughs[i], alpha=0.85)

    ax.set_yticks(y_pos + bar_h * len(_all_dfs) / 2)
    ax.set_yticklabels(available_feats, fontsize=9)
    ax.set_xlabel("Normalized mean (0 = min borough, 1 = max borough)")
    ax.set_title("Feature Means Comparison Across Boroughs (normalized)", fontsize=13)
    ax.legend(fontsize=10)

    plt.tight_layout()
    plt.savefig(f"{PLOTS_DIR}/05_feature_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {PLOTS_DIR}/05_feature_comparison.png")
else:
    print("No common feature columns found — skipping feature comparison.")

In [ ]:
# ── Prepare combined ML data for "Combined" panels ───

if not COMBINED_SHEETS:
    print("SKIP  Combined sheets (COMBINED_SHEETS = False)")
else:
    from sklearn.preprocessing import LabelEncoder, StandardScaler
    from sklearn.model_selection import train_test_split
    from sklearn.linear_model import LogisticRegression
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import confusion_matrix as sk_confusion_matrix
    import xgboost as xgb_lib
    import seaborn as sns
    import matplotlib.gridspec as gridspec
    sns.set(color_codes=True)

    _FEAT_COLS = ["amenity_density", "shop_density_km2",
                  "brand_ratio", "tourism_density", "landuse_entropy",
                  "amenity_ratio_food_drink", "avg_floors", "avg_yearbuilt",
                  "building_count", "total_bldg_area"]

    # Load and merge all boroughs' combined_grid.csv
    _feat_dfs = []
    for name, info in boroughs.items():
        _cp = info["csv_folder"] / "combined_grid.csv"
        if _cp.exists():
            _d = pd.read_csv(_cp, dtype={"cell_id": str})
            _d["borough"] = name
            _feat_dfs.append(_d)

    df_merged = pd.concat(_feat_dfs, ignore_index=True)
    df_merged["label"] = df_merged["zone_type"].map(_CLASS_MAP)
    df_merged = df_merged[df_merged["label"].notna()].reset_index(drop=True)

    X_comb = df_merged[_FEAT_COLS].fillna(df_merged[_FEAT_COLS].median())
    _enc_c = LabelEncoder()
    y_comb = _enc_c.fit_transform(df_merged["label"])
    _cnames = list(_enc_c.classes_)
    _scaler_c = StandardScaler()
    X_comb_sc = _scaler_c.fit_transform(X_comb)
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_comb_sc, y_comb, test_size=0.2, random_state=42, stratify=y_comb)

    _lr_c = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42).fit(X_tr, y_tr)
    _rf_c = RandomForestClassifier(class_weight="balanced", random_state=42).fit(X_tr, y_tr)
    _xgb_c = xgb_lib.XGBClassifier(random_state=14, eval_metric="mlogloss").fit(X_tr, y_tr)

    print(f"Combined: {len(df_merged)} cells from {len(boroughs)} boroughs")
    print(f"  LR={_lr_c.score(X_te, y_te):.3f}  XGB={_xgb_c.score(X_te, y_te):.3f}  RF={_rf_c.score(X_te, y_te):.3f}")

In [ ]:
# ── Generate combined sheets: 5 boroughs + Combined ──

if not COMBINED_SHEETS:
    print("SKIP  Combined sheets generation (COMBINED_SHEETS = False)")
else:
    def _make_sheet(plot_file, title, right_func):
        """Left: 5 borough thumbnail images. Right (large): combined panel."""
        b_list = list(boroughs.keys())
        fig = plt.figure(figsize=(38, 20))
        outer = gridspec.GridSpec(1, 2, figure=fig, width_ratios=[1, 1.3],
                                  wspace=0.15, left=0.02, right=0.98)
        fig.suptitle(title, fontsize=20, fontweight="bold", y=0.98)

        # Left side: 3x2 grid of borough thumbnails
        left = gridspec.GridSpecFromSubplotSpec(3, 2, subplot_spec=outer[0],
                                                hspace=0.12, wspace=0.06)
        pos = [(0,0),(0,1),(1,0),(1,1),(2,0)]
        for i, nm in enumerate(b_list[:5]):
            ax = fig.add_subplot(left[pos[i][0], pos[i][1]])
            ip = boroughs[nm]["outputs_folder"] / plot_file
            if ip.exists():
                ax.imshow(mpimg.imread(str(ip)))
            else:
                ax.text(0.5, 0.5, "Not found", ha="center", va="center",
                        transform=ax.transAxes, fontsize=12, color="gray")
            ax.set_title(nm, fontsize=13, fontweight="bold")
            ax.axis("off")
        fig.add_subplot(left[2, 1]).axis("off")

        # Right side: combined panel
        right_func(fig, outer[1])

        op = f"{PLOTS_DIR}/Combined Plots/combined_{plot_file}"
        plt.savefig(op, dpi=150, bbox_inches="tight")
        plt.close()
        print(f"  Saved: {op}")

    def _make_sheet_thumbs_only(plot_file, title):
        """All borough thumbnails only — no combined panel."""
        b_list = list(boroughs.keys())
        n = len(b_list)
        ncols = min(n, 3)
        nrows = math.ceil(n / ncols)
        fig, axes = plt.subplots(nrows, ncols, figsize=(10 * ncols, 8 * nrows))
        fig.suptitle(title, fontsize=20, fontweight="bold", y=1.0)
        if n == 1:
            axes = np.array([axes])
        axes = np.atleast_2d(axes)

        for i, nm in enumerate(b_list):
            r, c = divmod(i, ncols)
            ax = axes[r, c]
            ip = boroughs[nm]["outputs_folder"] / plot_file
            if ip.exists():
                ax.imshow(mpimg.imread(str(ip)))
            else:
                ax.text(0.5, 0.5, "Not found", ha="center", va="center",
                        transform=ax.transAxes, fontsize=12, color="gray")
            ax.set_title(nm, fontsize=15, fontweight="bold")
            ax.axis("off")

        for j in range(n, nrows * ncols):
            r, c = divmod(j, ncols)
            axes[r, c].axis("off")

        op = f"{PLOTS_DIR}/Combined Plots/combined_{plot_file}"
        plt.savefig(op, dpi=150, bbox_inches="tight")
        plt.close()
        print(f"  Saved: {op}")


    # ═══ 01 Countplot ═══
    def _r01(fig, spec):
        ax = fig.add_subplot(spec)
        counts = df_merged.groupby(["borough", "label"]).size().unstack(fill_value=0)
        cols = [c for c in CLASS_ORDER if c in counts.columns]
        order = [b for b in sorted(counts.index) if b in _KNOWN_BOROUGHS]
        order += [b for b in counts.index if b not in _KNOWN_BOROUGHS]
        counts.loc[order, cols].plot(kind="bar", ax=ax,
            color=[CLASS_COLS[c] for c in cols], alpha=0.85, width=0.8)
        ax.set_title("Combined — Zone Type Distribution", fontsize=16, fontweight="bold")
        ax.set_xlabel("")
        ax.set_ylabel("Cell count", fontsize=12)
        ax.tick_params(axis="x", rotation=25, labelsize=11)
        ax.legend(fontsize=12)
    print("Generating combined sheets...")
    _make_sheet("01_countplot_zone_type.png",
                "Zone Type Distribution — All Boroughs", _r01)


    # ═══ 02 Feature boxplots ═══
    def _r02(fig, spec):
        _feats = ["amenity_density", "shop_density_km2", "tourism_density",
                  "landuse_entropy", "avg_floors", "brand_ratio"]
        gs = gridspec.GridSpecFromSubplotSpec(2, 3, subplot_spec=spec,
                                              hspace=0.35, wspace=0.3)
        for i, feat in enumerate(_feats):
            ax = fig.add_subplot(gs[i // 3, i % 3])
            if feat in df_merged.columns:
                sns.boxplot(data=df_merged, x="label", y=feat, ax=ax,
                            palette=CLASS_COLS, order=CLASS_ORDER)
                ax.set_title(f"Combined — {feat}", fontsize=11, fontweight="bold")
                ax.set_xlabel("")
                ax.tick_params(axis="x", labelsize=9)
    _make_sheet("02_feature_boxplots.png",
                "Feature Boxplots — All Boroughs", _r02)


    # ═══ 03 Correlation heatmap ═══
    def _r03(fig, spec):
        # Get the right-side bounding box and inset the axes to avoid overlap
        bb = spec.get_position(fig)
        ax = fig.add_axes([bb.x0 + 0.04, bb.y0 + 0.02, bb.width - 0.06, bb.height - 0.06])
        corr = df_merged[_FEAT_COLS].corr()
        sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
                    square=True, ax=ax, vmin=-1, vmax=1, annot_kws={"size": 8})
        ax.set_title("Combined — Correlation Heatmap", fontsize=16, fontweight="bold")
    _make_sheet("03_correlation_heatmap.png",
                "Correlation Heatmap — All Boroughs", _r03)


    # ═══ 04–06 Confusion matrices (thumbnails only, no combined panel) ═══
    _make_sheet_thumbs_only("04_confusion_matrix_lr.png",
                            "Confusion Matrix (LR) — All Boroughs")
    _make_sheet_thumbs_only("05_confusion_matrix_xgb.png",
                            "Confusion Matrix (XGBoost) — All Boroughs")
    _make_sheet_thumbs_only("06_confusion_matrix_rf.png",
                            "Confusion Matrix (RF) — All Boroughs")


    # ═══ 07 Feature importance ═══
    def _r07(fig, spec):
        # Get the right-side bounding box and inset to leave room for y-labels
        bb = spec.get_position(fig)
        ax = fig.add_axes([bb.x0 + 0.07, bb.y0 + 0.02, bb.width - 0.09, bb.height - 0.06])
        imp = pd.Series(_rf_c.feature_importances_, index=_FEAT_COLS).sort_values(ascending=True)
        imp.plot(kind="barh", ax=ax, color="#4C72B0")
        ax.set_title("Combined — RF Feature Importance", fontsize=16, fontweight="bold")
        ax.set_xlabel("Importance", fontsize=12)
        ax.tick_params(axis="y", labelsize=11)
    _make_sheet("07_feature_importance_rf.png",
                "Feature Importance (RF) — All Boroughs", _r07)


    # ═══ 08 Heatmap ═══
    def _r08(fig, spec):
        ax = fig.add_subplot(spec)
        cmap_path = Path(PLOTS_DIR) / "02_combined_map.png"
        if cmap_path.exists():
            ax.imshow(mpimg.imread(str(cmap_path)))
        ax.set_title("Combined — All Boroughs", fontsize=16, fontweight="bold")
        ax.axis("off")
    _make_sheet("08_heatmap_predictions.png",
                "Heatmap Predictions — All Boroughs", _r08)


    # ═══ 09 Summary dashboard ═══
    def _r09(fig, spec):
        gs = gridspec.GridSpecFromSubplotSpec(2, 2, subplot_spec=spec,
                                              hspace=0.35, wspace=0.3)
        total = len(df_merged)

        # (0,0) Predicted distribution pie
        ax = fig.add_subplot(gs[0, 0])
        counts_pred = df_merged["label"].value_counts()
        sizes = [counts_pred.get(c, 0) for c in CLASS_ORDER if counts_pred.get(c, 0) > 0]
        labels = [f"{c}\n{counts_pred.get(c,0):,} ({100*counts_pred.get(c,0)/total:.1f}%)"
                  for c in CLASS_ORDER if counts_pred.get(c, 0) > 0]
        colors = [CLASS_COLS[c] for c in CLASS_ORDER if counts_pred.get(c, 0) > 0]
        ax.pie(sizes, labels=labels, colors=colors, startangle=90,
               textprops={"fontsize": 10}, wedgeprops={"edgecolor": "white", "linewidth": 1.5})
        ax.set_title("Combined — Class Distribution", fontsize=12, fontweight="bold")

        # (0,1) Per-borough accuracy
        ax = fig.add_subplot(gs[0, 1])
        b_names_sorted = list(boroughs.keys())
        accs = []
        for nm in b_names_sorted:
            df_b = boroughs[nm]["df"]
            df_b_label = df_b["zone_type"].map(_CLASS_MAP)
            accs.append((df_b["predicted_zone"] == df_b_label).mean())
        colors_acc = ["#2ca02c" if a >= 0.8 else "#ff7f0e" if a >= 0.7 else "#d62728" for a in accs]
        bars = ax.bar(range(len(b_names_sorted)), [a * 100 for a in accs], color=colors_acc, alpha=0.85)
        ax.set_xticks(range(len(b_names_sorted)))
        ax.set_xticklabels(b_names_sorted, rotation=30, ha="right", fontsize=9)
        ax.set_ylabel("Accuracy (%)")
        ax.set_ylim(0, 105)
        for i, v in enumerate(accs):
            ax.text(i, v * 100 + 1, f"{v:.1%}", ha="center", fontsize=9)
        ax.set_title("Per-Borough Accuracy", fontsize=12, fontweight="bold")

        # (1,0) Combined model accuracy
        ax = fig.add_subplot(gs[1, 0])
        models = ["LR", "XGBoost", "RF"]
        scores = [_lr_c.score(X_te, y_te), _xgb_c.score(X_te, y_te), _rf_c.score(X_te, y_te)]
        colors_m = ["#2ca02c" if s >= 0.8 else "#ff7f0e" for s in scores]
        ax.bar(models, [s * 100 for s in scores], color=colors_m, alpha=0.85)
        ax.set_ylim(0, 105)
        for i, s in enumerate(scores):
            ax.text(i, s * 100 + 1, f"{s:.1%}", ha="center", fontsize=11, fontweight="bold")
        ax.set_ylabel("Accuracy (%)")
        ax.set_title("Combined ML Accuracy", fontsize=12, fontweight="bold")

        # (1,1) Summary text
        ax = fig.add_subplot(gs[1, 1])
        ax.axis("off")
        lines = [f"COMBINED SUMMARY", f"{'='*30}",
                 f"Total cells: {total:,}", ""]
        for nm in b_names_sorted:
            lines.append(f"  {nm}: {len(boroughs[nm]['df']):,} cells")
        lines += ["", f"Combined ML (train on all):",
                  f"  LR:  {scores[0]:.1%}",
                  f"  XGB: {scores[1]:.1%}",
                  f"  RF:  {scores[2]:.1%}"]
        ax.text(0.05, 0.55, "\n".join(lines), transform=ax.transAxes,
                fontsize=12, fontfamily="monospace", va="center",
                bbox=dict(boxstyle="round,pad=0.5", facecolor="lightyellow", alpha=0.9))

    _make_sheet("09_summary_dashboard.png",
                "Summary Dashboard — All Boroughs", _r09)

    print(f"\nAll 9 combined sheets saved to {PLOTS_DIR}/Combined Plots/")

In [ ]:
# ── Summary table ────────────────────────────────────

summary_path = f"{PLOTS_DIR}/comparison_summary.csv"
df_summary.to_csv(summary_path, index=False, encoding="utf-8")
print(f"Saved: {summary_path}")

print(f"\n{'='*55}")
print(f"  COMPARISON COMPLETE")
print(f"{'='*55}")
print(f"  Boroughs compared: {', '.join(boroughs.keys())}")
print(f"  Total cells: {sum(len(info['df']) for info in boroughs.values())}")
print(f"  Output folder: {PLOTS_DIR}/")
print(f"{'='*55}")